In [1]:
# Import the libraries

import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

In [2]:
# Create a custom classification dataset
np.random.seed(42)
num_samples = 500

data = {
    "age": np.random.randint(18, 70, num_samples),                # years
    "income": np.random.randint(30000, 120000, num_samples),     # dollars
    "years_with_company": np.random.randint(0, 20, num_samples), # years
    "num_products": np.random.randint(1, 5, num_samples),        # products used
    "has_cr_card": np.random.randint(0, 2, num_samples),         # 0/1
    "is_active_member": np.random.randint(0, 2, num_samples),    # 0/1
}

df = pd.DataFrame(data)

In [3]:
# Target variable: churn (0 = stay, 1 = churn)
df["churn"] = (
    (df["age"] > 50).astype(int) +
    (df["income"] < 50000).astype(int) +
    (df["is_active_member"] == 0).astype(int)
)
df["churn"] = (df["churn"] > 1).astype(int)  # convert to 0/1

In [4]:
df.head()

,age,income,years_with_company,num_products,has_cr_card,is_active_member,churn
0,56,33343,5,1,0,0,1
1,69,43500,1,2,0,0,1
2,46,83222,19,3,1,0,0
3,32,59375,10,1,1,0,0
4,60,39662,3,3,0,1,1


In [5]:
# Separate the data
X = df.drop("churn", axis=1)
y = df["churn"]

In [6]:
# Scale features to be non-negative (required for Chi2)
# Chi2 requires non-negative values, that is why I am using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
# Apply selectKBest with Chi2
k = 3
selector = SelectKBest(score_func=chi2, k=k)
X_new = selector.fit_transform(X_scaled, y)

In [8]:
# get selected feature names
selected_features = X.columns[selector.get_support()]
print(f"Top Features: {selected_features}")

Top Features: Index(['age', 'income', 'is_active_member'], dtype='object')


In [9]:
# Get Chi Square Scores for all features
chi2_scores = selector.scores_
chi2_scores

array([1.56303185e+01, 4.87983348e+00, 3.60867351e-03, 2.82005230e-02,
       1.34454424e+00, 4.69841481e+01])

In [11]:
chi2_df = pd.DataFrame({
    "Feature": X.columns,
    "Chi2_Score": chi2_scores
})
chi2_df = chi2_df.sort_values(by="Chi2_Score", ascending=False)
print(chi2_df)

              Feature  Chi2_Score
5    is_active_member   46.984148
0                 age   15.630319
1              income    4.879833
4         has_cr_card    1.344544
3        num_products    0.028201
2  years_with_company    0.003609


### Train a simple Logistic Regression model on selected features

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [13]:
# Evaluate the model
print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Model Accuracy: 0.87
F1 Score: 0.7450980392156863
Confusion Matrix:
 [[68  3]
 [10 19]]
